# Instructions

* Due date is 10/29 midnight.
* Please check this original colab often to see if anything changes in the instructions. Last updated: 10/21 4:50pm.
* Make a copy of this colab, add your code, run all the cells and save output. Submit your .ipynb file with output. Don't submit a link to your colab, you must submit actual .ipynb file with output.
* You must train the model and show the test output. Points will be deducted if your model is not trained / produces run time errors. If you try to build a very large model and it doesn't finish training, than you would get points deducted, so make reasonable adjustments to keep the model size under check, do not train very small models, do not train very large models. If your training data is very large, feel free to sample it down, but do not sample down too much otherwise the model will not be trained properly.


Q1 CNN (10 Points)

Train a classifier for [animals](https://www.kaggle.com/datasets/likhon148/animal-data) (split this dataset into train and validation). Use standard data augmentation.

Train in two ways.

1. Create a model from scratch with 4 residual blocks and train that.

2. Get pretrained resnet-50 with include_top=False, add 1 new residual block on top, (and you would still need globalavg+dense at the top to predict labels), and train this.

Report accuracy for both the models.


# Training Part 1

In [ ]:
!kaggle datasets download likhon148/animal-data --unzip

import tensorflow as tf
import keras
from keras import layers, models, applications

for gpu in tf.config.experimental.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

path = "/content/animal_data"
img_height = 150
img_width = 150

# Train dataset for the path folder
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    path,
    validation_split=0.9,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=32,
    label_mode='categorical'
)

# Validation dataset for the path folder
validation_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    path,
    validation_split=0.1,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=32,
    label_mode='categorical'
)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomBrightness(0.1)
])

# Apply data augmentation to the training dataset only
train_dataset = train_dataset.map(lambda x, y: (data_augmentation(x, training=True), y))


# Define a simple residual block
def residual_block(x, filters, kernel_size=3):
    shortcut = x
    x = keras.layers.Conv2D(filters, kernel_size, padding='same')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)

    x = keras.layers.Conv2D(filters, kernel_size, padding='same')(x)
    x = keras.layers.BatchNormalization()(x)

    x = keras.layers.Add()([x, shortcut])
    x = keras.layers.Activation('relu')(x)

    return x

# Build model with 4 residual blocks
def create_resnet_model(input_shape):
    inputs = keras.layers.Input(shape=input_shape)

    x = keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu')(inputs)

    for _ in range(4):
        x = residual_block(x, filters=64)
        x = keras.layers.MaxPooling2D(pool_size=(2, 2))(x)

    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(128, activation='relu')(x)
    x = keras.layers.Dense(15, activation='softmax')(x)
    model = models.Model(inputs=inputs, outputs=x)
    return model


input_shape = (150, 150, 3)
model = create_resnet_model(input_shape)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train the model
epochs = 10
Train_history = model.fit(train_dataset, validation_data = validation_dataset, epochs=epochs, callbacks=[early_stopping])

if early_stopping.stopped_epoch > 0:
    print(f"Epoch {early_stopping.stopped_epoch + 1}: early stopping")
    print(f"Restoring model weights from the end of the best epoch: {early_stopping.best_epoch + 1}.")


Dataset URL: https://www.kaggle.com/datasets/likhon148/animal-data
License(s): MIT
 47% 17.0M/36.3M [00:00<00:00, 174MB/s]
100% 36.3M/36.3M [00:00<00:00, 225MB/s]
Found 1944 files belonging to 15 classes.
Using 195 files for training.
Found 1944 files belonging to 15 classes.
Using 194 files for validation.
Epoch 1/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 144s 20s/step - accuracy: 0.0719 - loss: 111.7002 - val_accuracy: 0.0567 - val_loss: 167.4298
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 138s 19s/step - accuracy: 0.0791 - loss: 125.5111 - val_accuracy: 0.0979 - val_loss: 78.8341
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 151s 21s/step - accuracy: 0.0856 - loss: 56.2305 - val_accuracy: 0.0670 - val_loss: 31.8635
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 139s 20s/step - accuracy: 0.0580 - loss: 21.8309 - val_accuracy: 0.0567 - val_loss: 15.7132
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 121s 17s/step - accuracy: 0.0850 - loss: 11.3821 - val_accuracy: 0.0773 - val_loss: 8.3819
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 160s 20s/step

In [ ]:
val_loss, val_accuracy = model.evaluate(validation_dataset)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 24s 3s/step - accuracy: 0.1368 - loss: 4.5479
Validation Loss: 4.2497
Validation Accuracy: 0.1546


In [ ]:
image_path = '/content/animal_data/Bear/Bear_10.jpg'
import cv2
import numpy as np
import os

image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image = cv2.resize(image, (150, 150))
image = np.expand_dims(image, axis=0)  # Add batch dimension

# Predict using the pretrained model
predictions = model.predict(image)
predicted_class = np.argmax(predictions, axis=1)

class_keys = sorted(os.listdir('/content/animal_data'))

# Map the predicted class index to the class name
predicted_class_name = class_keys[predicted_class[0]]
predicted_class_index = predictions[0].argmax()
print(f'The predicted class is: {predicted_class_name} with probability {predictions[0][predicted_class_index]:.2f}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 750ms/step
The predicted class is: Horse with probability 0.30


# Training Part 2

In [ ]:
!kaggle datasets download likhon148/animal-data --unzip

import tensorflow as tf
import keras
from keras import layers, models, applications

for gpu in tf.config.experimental.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

path = "/content/animal_data"
img_height = 224
img_width = 224

# Train dataset for the path folder
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    path,
    validation_split=0.9,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=32,
    label_mode='categorical'
)

# Validation dataset for the path folder
validation_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    path,
    validation_split=0.1,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=32,
    label_mode='categorical'
)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomBrightness(0.1)
])

# Apply data augmentation to the training dataset only
train_dataset = train_dataset.map(lambda x, y: (data_augmentation(x, training=True), y))

base_model = keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze base model layers
for layer in base_model.layers:
    layer.trainable = False

# Add custom residual block on top
def custom_residual_block(input_tensor):
    x = keras.layers.Conv2D(2048, (1, 1), padding='same')(input_tensor)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)

    x = keras.layers.Conv2D(2048, (3, 3), padding='same')(x)
    x = keras.layers.BatchNormalization()(x)

    # Skip connection
    x = keras.layers.Add()([x, input_tensor])
    x = keras.layers.Activation('relu')(x)
    return x

# Add layers to the model
x = base_model.output
x = custom_residual_block(x)  # Additional residual block
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dense(256, activation='relu')(x)
num_classes = train_dataset.element_spec[1].shape[-1]
x = keras.layers.Dense(num_classes, activation='softmax')(x)

model1 = models.Model(inputs=base_model.input, outputs=x)

early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

model1.compile(optimizer=tf.keras.optimizers.Adam(),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
history = model1.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=10, callbacks=[early_stopping]  # Adjust the number of epochs as needed
)

if early_stopping.stopped_epoch > 0:
    print(f"Epoch {early_stopping.stopped_epoch + 1}: early stopping")
    print(f"Restoring model weights from the end of the best epoch: {early_stopping.best_epoch + 1}.")

Dataset URL: https://www.kaggle.com/datasets/likhon148/animal-data
License(s): MIT
 55% 20.0M/36.3M [00:00<00:00, 207MB/s]
100% 36.3M/36.3M [00:00<00:00, 242MB/s]
Found 1944 files belonging to 15 classes.
Using 195 files for training.
Found 1944 files belonging to 15 classes.
Using 194 files for validation.
Epoch 1/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 224s 32s/step - accuracy: 0.2738 - loss: 3.4371 - val_accuracy: 0.3299 - val_loss: 12.6331
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 260s 31s/step - accuracy: 0.6554 - loss: 1.1282 - val_accuracy: 0.5567 - val_loss: 6.1313
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 233s 26s/step - accuracy: 0.8828 - loss: 0.4641 - val_accuracy: 0.5773 - val_loss: 4.0796
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 234s 32s/step - accuracy: 0.9053 - loss: 0.3566 - val_accuracy: 0.5979 - val_loss: 3.5452
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 258s 31s/step - accuracy: 0.9460 - loss: 0.2044 - val_accuracy: 0.6701 - val_loss: 3.2360
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 261s 31s/step - accuracy

In [ ]:
val_loss, val_accuracy = model1.evaluate(validation_dataset)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 52s 7s/step - accuracy: 0.7067 - loss: 2.8820
Validation Loss: 3.2307
Validation Accuracy: 0.7010


In [ ]:
image_path = '/content/animal_data/Bear/Bear_10.jpg'
import cv2
import numpy as np
import os

image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image = cv2.resize(image, (224, 224))
image = np.expand_dims(image, axis=0)  # Add batch dimension

# Predict using the pretrained model
predictions = model1.predict(image)
predicted_class = np.argmax(predictions, axis=1)

class_keys = sorted(os.listdir('/content/animal_data'))

# Map the predicted class index to the class name
predicted_class_name = class_keys[predicted_class[0]]
predicted_class_index = predictions[0].argmax()
print(f'The predicted class is: {predicted_class_name} with probability {predictions[0][predicted_class_index]:.2f}')


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
The predicted class is: Bear with probability 1.00
